In [1]:
import pandas as pd
from cpdb_api import request

# Using the cpdb api to collect data on the following countries.

country_list = ["RWA", # Rwanda: Primary case study (Unitary, High Vulnerability)
  "DEU", # Germany: Global North leader (Federal, High Mitigation Stringency)
  "IND", # India: Global South power (Federal, High Growth/Sectoral Complexity)
  "BRA", # Brazil: Land-Use Focus (Federal, High Biodiversity/Adaptation Significance)
  "ZAF", # South Africa: Just Transition Model (Unitary, Coal-to-Green Transition focus)
  "JPN", # Japan: Vulnerable North (Unitary, Technology-led Adaptation)
  "CAN"  # Canada: Federal Resource Exporter (Federal, Mitigation/Adaptation tension)
]

def fetch_country_bulk(country):
    r = request.Request()
    r.set_country(country)
    try:
        return r.issue()
    except Exception as e:
        print(f"Failed for {country}: {e}")
        return None

results = [fetch_country_bulk(c) for c in country_list]
cpdb_master_df = pd.concat([res for res in results if res is not None], ignore_index=True)
print(f"Data Loaded: {len(cpdb_master_df)} rows.")

Data Loaded: 1292 rows.


In [ ]:
# Saving the data
save_path = "../data/cpdb_master.csv"
cpdb_master_df.to_csv(save_path, index=False)
print(f"Data saved to {save_path}.")


Data saved to ../data/cpdb_master.csv.


In [ ]:
# Classification logic for initial labeling of policies as "Progressive", "Regressive", or "Unclassified/Neutral". This is a very basic start

import pandas as pd

def initial_classifier(row):
    # Convert description 
    desc = str(row['policy_description']).lower()
    sector = str(row['sector']).lower()
    
    # Need to think about this logic... how would we classify this?
    # Logic for "Progressive" 
    progressive_keywords = ['subsidy', 'grant', 'public transport', 'rebate', 'incentive', 'funding']
    
    # Logic for "Regressive" 
    regressive_keywords = ['tax', 'levy', 'carbon price', 'fee', 'duty', 'removal of subsidy']

    # Check for Progressive signs
    if any(word in desc for word in progressive_keywords):
        return "Likely Progressive"
    
    # Check for Regressive signs
    if any(word in desc for word in regressive_keywords):
        return "Likely Regressive"
    
    return "Unclassified / Neutral"

# Apply to DataFrame
cpdb_master_df['initial_label'] = cpdb_master_df.apply(initial_classifier, axis=1)

# See how it did
print(cpdb_master_df['initial_label'].value_counts())

save_path = "../data/cpdb_master_with_labels.csv"
cpdb_master_df.to_csv(save_path, index=False)
print(f"Data with initial labels saved to {save_path}.")



In [ ]:
print(cpdb_master_df[['policy_description', 'initial_label']].head(20))

In [5]:
# just get the country, sector, description, and label for the next step of manual review.
review_df = cpdb_master_df[['country', 'sector', 'policy_description', 'initial_label']]
review_save_path = "../data/cpdb_review_subset.csv"
review_df.to_csv(review_save_path, index=False)
print(f"Subset for manual review saved to {review_save_path}.") 

Subset for manual review saved to ../data/cpdb_review_subset.csv.


In [7]:
# Getting rid of any empty descriptions
# if they are empty they are empty not n/a, so we can just check for that.
cpdb_master_df = cpdb_master_df[cpdb_master_df['policy_description'].str.strip() != '']
print(f"After removing empty descriptions: {len(cpdb_master_df)} rows remain.")


After removing empty descriptions: 733 rows remain.


In [10]:
# A more complex classifier that also considers the sector and country context. This is still very basic and would need a lot of refinement, but it's a start.

import pandas as pd

def classify_policy_v1(row):
    # Combine the three fields into one searchable string
    text_blob = f"{row['policy_name']} {row['policy_instrument']} {row['policy_description']}".lower()
    
    # Define weights
    progressive_terms = {
        'subsidy': 2, 'grant': 2, 'investment': 1, 'social': 3, 
        'rebate': 2, 'low-income': 3, 'public transit': 2, 'fund': 1
    }
    
    regressive_terms = {
        'tax': 2, 'levy': 2, 'carbon price': 2, 'fee': 1, 
        'duty': 2, 'excise': 2, 'removal': 1
    }
    
    score = 0
    
    # Calculate score
    for term, weight in progressive_terms.items():
        if term in text_blob:
            score += weight
            
    for term, weight in regressive_terms.items():
        if term in text_blob:
            score -= weight

    # Return classification
    if score > 0:
        return "Likely Progressive"
    elif score < 0:
        return "Likely Regressive"
    else:
        return "Neutral/Unclassified"


cpdb_master_df['classification'] = cpdb_master_df.apply(classify_policy_v1, axis=1)
print(cpdb_master_df['classification'].value_counts())
save_path = "../data/cpdb_master_with_classification.csv"

cpdb_master_df.to_csv(save_path, index=False)
print(f"Data with classifications saved to {save_path}.")

classification
Neutral/Unclassified    411
Likely Progressive      210
Likely Regressive       112
Name: count, dtype: int64
Data with classifications saved to ../data/cpdb_master_with_classification.csv.
